# California Housing — Case Study (Regression)

**Objective:** Predict median house value (regression).

**Dataset:** California Housing (sklearn `fetch_california_housing()`; downloaded/cached automatically).

**Experimental protocol (mandatory):**
1. Train/test split **once** at the beginning.
2. Use **cross-validation only on the training set** for model comparison and selection.
3. Evaluate the **baseline (mean predictor)** first.
4. Compare advanced models against the baseline using the same protocol.
5. Retrain the chosen model on the full training set; evaluate **exactly once** on the held-out test set.
6. Report and justify metric choice (MAE vs RMSE vs R²); residual analysis.

In [ ]:
# Imports and configuration
import numpy as np
import pandas as pd
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline

from src.config import RANDOM_STATE
from src.data_loading import load_housing
from src.preprocessing import make_regression_preprocessor
from src.splitting import split_regression, make_cv_regression
from src.metrics import reg_metrics
from src.evaluation import evaluate_regression_cv

np.random.seed(RANDOM_STATE)

## 1. Load data

In [ ]:
df = load_housing()
print("Shape:", df.shape)
df.head()

In [ ]:
df.info()

## 2. Exploratory Data Analysis (EDA)

Look at the data first — before any preprocessing, splitting, or training.

In [ ]:
# Missing values (California Housing has no missing values by design)
print("Missing per column:", df.isnull().sum().sum())

In [ ]:
# Target: MedHouseValue (median house value in $100k)
import matplotlib.pyplot as plt
df["MedHouseVal"].hist(bins=50, edgecolor="black", alpha=0.7)
plt.title("MedHouseVal (target)")
plt.xlabel("MedHouseVal")
plt.show()
print(df["MedHouseVal"].describe())

In [ ]:
# Features: all numeric (no categoricals in this dataset)
print("Feature summary:")
print(df.drop(columns=["MedHouseVal"]).describe())

In [ ]:
# Example: target vs median income
df.plot.scatter(x="MedInc", y="MedHouseVal", alpha=0.2, figsize=(6, 4))
plt.title("MedHouseVal vs MedInc")
plt.tight_layout()
plt.show()

## 3. Define target and features, prepare data

**Target:** `MedHouseVal` (median house value in $100,000). **Features:** all other columns (MedInc, HouseAge, AveRooms, AveBedrms, Population, AveOccup, Latitude, Longitude). This dataset is all numeric — no one-hot encoding needed.

In [ ]:
y = df["MedHouseVal"]
X = df.drop(columns=["MedHouseVal"])
print("X shape:", X.shape)
print("y (target) describe:\n", y.describe())

## 4. Train/test split (once)

We split **once** and do not use the test set until the final evaluation.

In [ ]:
X_train, X_test, y_train, y_test = split_regression(X, y)
print("Train size:", len(y_train), "| Test size:", len(y_test))
cv = make_cv_regression()

## 5. Preprocessing

Scale numeric features (StandardScaler). We fit on the training set only and use the same fitted preprocessor for test. This dataset has no categorical columns.

In [ ]:
preprocessor = make_regression_preprocessor(X_train)
# Pipeline(preprocessor, model) so that CV fits preprocessor per fold (no leakage)

## 6. Baseline: mean predictor

Before any complex model, we evaluate a **baseline** (regression: predict the mean of the training target). All other models must be compared against this using the same protocol.

In [ ]:
baseline_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("reg", DummyRegressor(strategy="mean")),
])
baseline_cv_metrics = evaluate_regression_cv(baseline_pipeline, X_train, y_train, cv)
print("Baseline CV metrics (on train):", baseline_cv_metrics)

In [ ]:
# Retrain baseline on full training set; evaluate exactly once on test set
baseline_pipeline.fit(X_train, y_train)
y_pred_baseline = baseline_pipeline.predict(X_test)
baseline_test_metrics = reg_metrics(y_test, y_pred_baseline)
print("Baseline test metrics (final, one-time evaluation):", baseline_test_metrics)

## 7. Model comparison (CV on training set only)

**Define model candidates first, then run CV.** We pre-define the second candidate (Ridge regression) before the CV cell; then we run all candidates through the same CV and compare.

In [ ]:
# Candidate 1: Baseline (already defined above). Candidate 2: Ridge — instantiate before CV
ridge_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("reg", Ridge(random_state=RANDOM_STATE)),
])

**Before the comparison CV — your turn (case study):** Add at least 2 more model candidates (e.g. Random Forest, Gradient Boosting, k-NN). Define each model and a hyperparameter search space; run tuning on the training set only (e.g. GridSearchCV or RandomizedSearchCV). Include the best-tuned pipelines in the comparison CV below. Once you have all candidates defined (and tuned), the next step is the comparison CV.

**Comparison CV:** Run all candidates through the **same CV** on the training set and compare metrics to select the best.

In [ ]:
# Run all pre-defined candidates through the same CV
baseline_cv_metrics = evaluate_regression_cv(baseline_pipeline, X_train, y_train, cv)
ridge_cv_metrics = evaluate_regression_cv(ridge_pipeline, X_train, y_train, cv)

cv_comparison = pd.DataFrame({
    "Baseline (mean)": baseline_cv_metrics,
    "Ridge": ridge_cv_metrics,
}).T
print("CV metrics (on train) — model candidates compared:")
display(cv_comparison)

## 8. Retrain selected model and final test evaluation

We choose the best configuration based on CV (here: Ridge). We **retrain** on the **full training set** and evaluate **exactly once** on the test set.

In [ ]:
ridge_pipeline.fit(X_train, y_train)
y_pred_ridge = ridge_pipeline.predict(X_test)
ridge_test_metrics = reg_metrics(y_test, y_pred_ridge)
print("Ridge test metrics (final, one-time evaluation):", ridge_test_metrics)

In [ ]:
# Compare baseline vs Ridge on test set
pd.DataFrame({
    "Baseline (mean)": baseline_test_metrics,
    "Ridge": ridge_test_metrics,
}).T

## 9. Metric justification

For regression, common metrics are:

- **MAE** (mean absolute error): average absolute error; same units as the target; robust to outliers.
- **RMSE** (root mean squared error): penalizes large errors more; same units as the target.
- **R²** (coefficient of determination): fraction of variance explained; 0 = no better than predicting the mean, 1 = perfect.

Justify your choice depending on the goal (e.g. if large errors are especially costly, RMSE may be preferred; if interpretability in dollars matters, MAE is intuitive).

## 10. Error analysis: residuals

Inspect residuals (actual − predicted) on the test set: distribution, residual vs predicted plot. Use the test set only for this descriptive analysis after the single final evaluation.

In [ ]:
# Residuals: actual - predicted (on test set)
residuals = y_test - y_pred_ridge
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].hist(residuals, bins=50, edgecolor="black", alpha=0.7)
axes[0].set_xlabel("Residual")
axes[0].set_title("Residual distribution")
axes[1].scatter(y_pred_ridge, residuals, alpha=0.3)
axes[1].axhline(0, color="red", linestyle="--")
axes[1].set_xlabel("Predicted")
axes[1].set_ylabel("Residual")
axes[1].set_title("Residual vs predicted")
plt.tight_layout()
plt.show()

---

**Next steps (for your case study):** Add more models (e.g. Random Forest, Gradient Boosting), tune hyperparameters via CV on the training set only, deepen residual analysis (e.g. by region or income), and justify your final metric and model choice in writing.